<a href="https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [10]:
question = """
Research question:
Which pre-March search and engagement signals are associated with stronger
March page-performance outcomes, and can these signals support a transparent
ranking of pages for refresh or review?

Decision supported:
The analysis is intended to help a content/SEO team prioritize which pages
deserve review first when review capacity is limited. The ranking is decision
support, not a claim that refreshing a page will cause better performance.
"""

print(question)


Research question:
Which pre-March search and engagement signals are associated with stronger
March page-performance outcomes, and can these signals support a transparent
ranking of pages for refresh or review?

Decision supported:
The analysis is intended to help a content/SEO team prioritize which pages
deserve review first when review capacity is limited. The ranking is decision
support, not a claim that refreshing a page will cause better performance.



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [11]:
data_contract = """
Data:
Source: FlyRank internship warehouse, fact_content_daily_performance.

Feature release:
February 2026 (2026-02-01 to 2026-02-28), used as the pre-label feature window.
Verified February rows: 7,355,108.

Outcome/label release:
March 2026 (2026-03-01 to 2026-03-31), used as the outcome window.
Verified March rows: 9,841,378.

Unit of analysis:
One content item for one client on one report date
(client_hash_id × content_hash_id × report_date).

Selected data:
GSC search signals, GA4 performance signals, client/content identifiers,
report date, and data-availability fields.

Excluded from the initial model:
Traffic-source fields and AI-referral fields were excluded because they were
not necessary for the first version of the analysis and may contain
sparse/zero-heavy signals.

Public-safety:
No client names, private queries, raw exports, credentials, or private URLs
are included in the analysis or paper.

The February and March windows are kept separate so that model features use
only information available before the March outcome period.
"""

print(data_contract)


Data:
Source: FlyRank internship warehouse, fact_content_daily_performance.

Feature release:
February 2026 (2026-02-01 to 2026-02-28), used as the pre-label feature window.
Verified February rows: 7,355,108.

Outcome/label release:
March 2026 (2026-03-01 to 2026-03-31), used as the outcome window.
Verified March rows: 9,841,378.

Unit of analysis:
One content item for one client on one report date
(client_hash_id × content_hash_id × report_date).

Selected data:
GSC search signals, GA4 performance signals, client/content identifiers,
report date, and data-availability fields.

Excluded from the initial model:
Traffic-source fields and AI-referral fields were excluded because they were
not necessary for the first version of the analysis and may contain
sparse/zero-heavy signals.

Public-safety:
No client names, private queries, raw exports, credentials, or private URLs
are included in the analysis or paper.

The February and March windows are kept separate so that model features use
o

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [12]:
methodology = """
Methodology:

Features:
The model uses pre-March search and engagement signals from February,
including GSC impressions, GSC clicks, GSC average position, and GA4
pageviews, sessions, engaged sessions, and total engagement time.
Data-availability fields are retained as context for interpreting missing
signals.

Label:
The March outcome is defined from March page-performance activity. The
analysis uses the March window only as the outcome period and does not use
March information as an input feature.

Baseline:
A transparent rule-based baseline ranks pages using February GSC exposure,
clicks, and average position. The baseline uses simple conditions and no
fitted weights.

Model:
A machine-learning model is compared against the transparent baseline using
the same February feature window and March outcome definition.

Validation:
Model performance is evaluated on a held-out validation split so that the
reported comparison is based on data not used to fit the model.

Leakage checks:
Feature construction is restricted to information available by the end of
February. March outcome fields, future-window information, and product flags
are not used as model inputs. Client/content identifiers are used for
grouping and traceability rather than as predictive signals.

Interpretation:
Results are framed as observed associations and decision-support signals.
The analysis does not claim that a model score proves that refreshing a page
will cause better future performance.
"""

print(methodology)


Methodology:

Features:
The model uses pre-March search and engagement signals from February,
including GSC impressions, GSC clicks, GSC average position, and GA4
pageviews, sessions, engaged sessions, and total engagement time.
Data-availability fields are retained as context for interpreting missing
signals.

Label:
The March outcome is defined from March page-performance activity. The
analysis uses the March window only as the outcome period and does not use
March information as an input feature.

Baseline:
A transparent rule-based baseline ranks pages using February GSC exposure,
clicks, and average position. The baseline uses simple conditions and no
fitted weights.

Model:
A machine-learning model is compared against the transparent baseline using
the same February feature window and March outcome definition.

Validation:
Model performance is evaluated on a held-out validation split so that the
reported comparison is based on data not used to fit the model.

Leakage checks:
Feat

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [13]:
# Section 4: Results vs baseline

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

# Load only the columns needed for the capstone comparison.
needed_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# February feature data
feb_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

feb = pd.read_parquet(feb_file, columns=needed_cols)

# March outcome data
mar_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

mar = pd.read_parquet(mar_file, columns=needed_cols)

# Aggregate daily observations to one row per client-content pair.
# February = features; March = outcome.
group_cols = ["client_hash_id", "content_hash_id"]

feb_page = (
    feb.groupby(group_cols, as_index=False)
       .agg(
           feb_impressions=("gsc_impressions", "sum"),
           feb_clicks=("gsc_clicks", "sum"),
           feb_avg_position=("gsc_avg_position", "mean")
       )
)

mar_page = (
    mar.groupby(group_cols, as_index=False)
       .agg(
           march_clicks=("gsc_clicks", "sum")
       )
)

# Join pre-label features to the March outcome.
results_df = feb_page.merge(
    mar_page,
    on=group_cols,
    how="inner"
)

# Replace unavailable feature values with neutral values for this baseline.
results_df["feb_impressions"] = results_df["feb_impressions"].fillna(0)
results_df["feb_clicks"] = results_df["feb_clicks"].fillna(0)

# Position is only used when it is available.
results_df["position_signal"] = (
    results_df["feb_avg_position"].notna()
    & (results_df["feb_avg_position"] <= 20)
)

# Transparent baseline score: same logic as ML-07.
results_df["baseline_score"] = (
    2 * (results_df["feb_impressions"] > 0).astype(int)
    + 2 * (results_df["feb_clicks"] > 0).astype(int)
    + results_df["position_signal"].astype(int)
)

# Rank baseline.
results_df["baseline_rank"] = (
    results_df["baseline_score"]
    .rank(method="first", ascending=False)
)

# Simple interpretable model:
# use only February information to predict March clicks.
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

model_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position"
]

model_df = results_df.dropna(
    subset=["feb_avg_position", "march_clicks"]
).copy()

X = model_df[model_features]
y = model_df["march_clicks"]

# Hold-out split for an honest model evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = HistGradientBoostingRegressor(
    max_iter=100,
    max_leaf_nodes=15,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Model validation results")
print("Rows used:", len(model_df))
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("MAE:", round(mean_absolute_error(y_test, predictions), 4))
print("R2:", round(r2_score(y_test, predictions), 4))

# Compare ranking quality on the same held-out rows.
comparison = model_df.loc[X_test.index].copy()
comparison["model_score"] = predictions

comparison["model_rank"] = (
    comparison["model_score"]
    .rank(method="first", ascending=False)
)

comparison["actual_rank"] = (
    comparison["march_clicks"]
    .rank(method="first", ascending=False)
)

# Top-100 overlap gives an intuitive ranking comparison.
k = min(100, len(comparison))

baseline_top = set(
    comparison.nsmallest(k, "baseline_rank").index
)

model_top = set(
    comparison.nlargest(k, "model_score").index
)

overlap = len(baseline_top & model_top)

print("\nRanking comparison")
print("Top-k:", k)
print("Baseline/model top-k overlap:", overlap)
print("Overlap rate:", round(overlap / k, 4))

# Honest result summary.
baseline_mean = comparison[
    comparison.index.isin(baseline_top)
]["march_clicks"].mean()

model_mean = comparison[
    comparison.index.isin(model_top)
]["march_clicks"].mean()

print("\nTop-k outcome comparison")
print("Baseline top-k mean March clicks:", round(baseline_mean, 4))
print("Model top-k mean March clicks:", round(model_mean, 4))

Model validation results
Rows used: 145279
Train rows: 116223
Test rows: 29056
MAE: 3.0619
R2: 0.5686

Ranking comparison
Top-k: 100
Baseline/model top-k overlap: 1
Overlap rate: 0.01

Top-k outcome comparison
Baseline top-k mean March clicks: 11.61
Model top-k mean March clicks: 215.0


Results vs baseline: The model was evaluated on a held-out test set of 29,056 page-level observations. The model achieved an MAE of 3.0619 and an R² of 0.5686. In the top-100 ranking comparison, only 1% of the pages overlapped with the transparent baseline. The model-selected top 100 had a mean March click count of 215.0, compared with 11.61 for the baseline-selected top 100. This indicates that the model identified a substantially different high-opportunity set and, within this evaluation, concentrated pages with higher observed March click activity. These results are directional and should not be interpreted as causal evidence that the model or a refresh action produces higher traffic.

## 5. Limitations

*What this work cannot claim.*

In [14]:
limitations = """
Limitations:

1. This analysis measures association and predictive performance, not causality.
A high model score does not prove that refreshing a page will cause higher
future traffic or clicks.

2. The outcome is based on observed March GSC clicks. It reflects what was
measured during the outcome window and may also be affected by factors outside
the model, such as seasonality, search-engine changes, competition, or changes
to the page.

3. The model uses a February-to-March temporal split, but this is still a
single feature/outcome release. Results may not generalize to other months,
clients, or future periods.

4. GSC data is sparse and unevenly available across clients. Missingness and
data availability can therefore affect both model inputs and interpretation.

5. The model uses a limited set of transparent GSC signals. Additional
context such as content quality, query intent, SERP features, backlinks, and
business priorities is not captured.

6. The top-ranked pages should therefore be treated as review candidates,
not guaranteed refresh recommendations.

7. The baseline and model are decision-support tools. Human review is still
required before taking a content or SEO action.
"""

print(limitations)


Limitations:

1. This analysis measures association and predictive performance, not causality.
A high model score does not prove that refreshing a page will cause higher
future traffic or clicks.

2. The outcome is based on observed March GSC clicks. It reflects what was
measured during the outcome window and may also be affected by factors outside
the model, such as seasonality, search-engine changes, competition, or changes
to the page.

3. The model uses a February-to-March temporal split, but this is still a
single feature/outcome release. Results may not generalize to other months,
clients, or future periods.

4. GSC data is sparse and unevenly available across clients. Missingness and
data availability can therefore affect both model inputs and interpretation.

5. The model uses a limited set of transparent GSC signals. Additional
context such as content quality, query intent, SERP features, backlinks, and
business priorities is not captured.

6. The top-ranked pages should ther

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [15]:
import os

os.makedirs("work/outputs", exist_ok=True)

recommendations[recommendation_cols].head(100).to_csv(
    "work/outputs/capstone_ranked_recommendations.csv",
    index=False
)

print("Saved successfully:")
print("work/outputs/capstone_ranked_recommendations.csv")

Saved successfully:
work/outputs/capstone_ranked_recommendations.csv


In [16]:
# Section 6: Ranked recommendations

# Generate model scores for all eligible pages using February features.
recommendations = model_df.copy()

recommendations["model_score"] = model.predict(
    recommendations[model_features]
)

# Rank highest predicted March click opportunity first.
recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1, len(recommendations) + 1
)

# Transparent action labels.
recommendations["action"] = np.select(
    [
        recommendations["model_score"] >= recommendations["model_score"].quantile(0.90),
        recommendations["model_score"] >= recommendations["model_score"].quantile(0.75)
    ],
    [
        "high_priority_review",
        "review"
    ],
    default="monitor"
)

# Reason code based on the strongest February signal.
recommendations["reason_code"] = np.select(
    [
        (recommendations["feb_clicks"] > 0) &
        (recommendations["feb_impressions"] > 0) &
        (recommendations["feb_avg_position"] <= 20),

        (recommendations["feb_impressions"] > 0) &
        (recommendations["feb_clicks"] > 0)
    ],
    [
        "visible_with_clicks_and_good_position",
        "visible_with_clicks"
    ],
    default="limited_historical_signal"
)

recommendation_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "model_score",
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "march_clicks"
]

print("Top 20 ranked recommendations:")
display(
    recommendations[recommendation_cols].head(20)
)

# Save the recommendation artifact.
os.makedirs("work/outputs", exist_ok=True)

recommendations[recommendation_cols].head(100).to_csv(
    "work/outputs/capstone_ranked_recommendations.csv",
    index=False
)

print(
    "\nSaved:",
    "work/outputs/capstone_ranked_recommendations.csv"
)

Top 20 ranked recommendations:


,rank,client_hash_id,content_hash_id,action,reason_code,model_score,feb_impressions,feb_clicks,feb_avg_position,march_clicks
0,1,client_62f4a7e64f5e0096,content_44d284c2ad861171,high_priority_review,visible_with_clicks_and_good_position,762.887404,30752.0,220.0,2.545156,324
1,2,client_62f4a7e64f5e0096,content_c556c7369fb2fd06,high_priority_review,visible_with_clicks_and_good_position,762.887404,36689.0,161.0,2.570129,201
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,high_priority_review,visible_with_clicks_and_good_position,762.887404,119854.0,490.0,2.594817,1480
3,4,client_e547b89c05043229,content_055164e0e5126e87,high_priority_review,visible_with_clicks_and_good_position,762.887404,34470.0,204.0,2.523652,148
4,5,client_e547b89c05043229,content_eadb33b5df496f4a,high_priority_review,visible_with_clicks_and_good_position,762.887404,94852.0,1123.0,2.559384,5668
5,6,client_73cda7b4e4f265ea,content_804019fba2e15fe2,high_priority_review,visible_with_clicks_and_good_position,531.056795,72607.0,396.0,4.203246,376
6,7,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,high_priority_review,visible_with_clicks_and_good_position,531.056795,154502.0,2508.0,4.240872,2446
7,8,client_20259bd6705d81d4,content_5fa2737c68998c2e,high_priority_review,visible_with_clicks_and_good_position,531.056795,31428.0,249.0,4.238658,1053
8,9,client_62f4a7e64f5e0096,content_b64fb5a4b3f9b7aa,high_priority_review,visible_with_clicks_and_good_position,531.056795,65715.0,125.0,4.242257,102
9,10,client_62f4a7e64f5e0096,content_f54d72a322dab771,high_priority_review,visible_with_clicks_and_good_position,493.882896,47689.0,122.0,2.663211,107



Saved: work/outputs/capstone_ranked_recommendations.csv


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [17]:
artifacts = """
Artifacts produced for the capstone paper:

1. Baseline ranking:
   work/outputs/baseline_action_score.csv

2. Capstone ranked recommendations:
   work/outputs/capstone_ranked_recommendations.csv

3. Model evaluation:
   - Test rows: 29,056
   - MAE: 3.0619
   - R²: 0.5686

4. Ranking comparison:
   - Top-100 baseline/model overlap: 1%
   - Baseline top-100 mean March clicks: 11.61
   - Model top-100 mean March clicks: 215.0

5. Recommendation table:
   The paper should embed the ranked recommendation output with
   anonymized client/content identifiers, model score, action,
   reason code, February signals, and observed March clicks.

6. Methodology evidence:
   The paper should describe the February feature window,
   March outcome window, temporal validation, leakage checks,
   baseline comparison, and limitations.
"""

print(artifacts)


Artifacts produced for the capstone paper:

1. Baseline ranking:
   work/outputs/baseline_action_score.csv

2. Capstone ranked recommendations:
   work/outputs/capstone_ranked_recommendations.csv

3. Model evaluation:
   - Test rows: 29,056
   - MAE: 3.0619
   - R²: 0.5686

4. Ranking comparison:
   - Top-100 baseline/model overlap: 1%
   - Baseline top-100 mean March clicks: 11.61
   - Model top-100 mean March clicks: 215.0

5. Recommendation table:
   The paper should embed the ranked recommendation output with
   anonymized client/content identifiers, model score, action,
   reason code, February signals, and observed March clicks.

6. Methodology evidence:
   The paper should describe the February feature window,
   March outcome window, temporal validation, leakage checks,
   baseline comparison, and limitations.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
